# v0.50.0 — Official SDK 3.0 and SurrealDB 4.x

## 1. Intro

Surreal ORM Lite 0.50.0 runs on the official SurrealDB Python SDK 3.0 (`surrealdb[pydantic]>=3.0.0,<4.0.0`) and
supports the server lines that SDK supports: **SurrealDB 2.x, 3.x and 4.x**. The ORM's minor series follows the SDK
major: `0.50.x` is the SDK 3.0 line, `0.40.x` the SDK 2.0 line (deprecated, security fixes only).

Models, querysets, transactions and migrations keep their API. This notebook walks what changes
([migration note](https://github.com/EulogySnowfall/SurrealDB-ORM-lite/blob/main/docs/migrating-to-0.50.md)):

- values read through the SDK 3.0: datetimes keep their nanoseconds, `Duration(0)`, native sets, `NONE` as `None`;
- `raw_query()` returns the rows of every statement;
- `first()` returns the record with the smallest id on SurrealDB 3.x, 4.x and the embedded engine;
- the transaction strategy of each line;
- the embedded engine, a SurrealDB 4.0 core (`pip install "surreal-orm-lite[embedded]"`);
- migrations on SurrealDB 4.x: `FieldChangeRefusedError`, and the `RawSQL` that converts the rows.

The notebook runs top to bottom on SurrealDB 2.x, 3.x and 4.x. The outputs below come from a SurrealDB 3.x server;
§7 also runs on the embedded engine, which follows the SurrealDB 4.x rule, so both outcomes are shown.

## 2. Connect

`SURREALDB_HOST` and `SURREALDB_PORT` choose the server (default `localhost:8000`), over WebSocket. The notebook works
in a database of its own, `sdk3_demo`, removed first so that it can be re-executed in place. `cached_major()` reads the
server's major version once per client.

In [1]:
import os
from importlib.metadata import version

from surreal_orm_lite import SurrealDBConnectionManager
from surreal_orm_lite.time_travel import cached_major

HOST = os.environ.get("SURREALDB_HOST", "localhost")
PORT = os.environ.get("SURREALDB_PORT", "8000")
# WebSocket (.../rpc), as the other notebooks.
SurrealDBConnectionManager.set_connection(
    url=f"ws://{HOST}:{PORT}/rpc",
    user="root",
    password="root",
    namespace="examples",
    database="sdk3_demo",
)
client = await SurrealDBConnectionManager.get_client()
await client.query("REMOVE DATABASE IF EXISTS sdk3_demo; DEFINE DATABASE sdk3_demo;")
LINE = await cached_major(client)
print("Connection configured:", SurrealDBConnectionManager.is_connection_set())
print("surrealdb SDK:", version("surrealdb").split(".")[0] + ".x")
print(f"SurrealDB server line: {LINE}.x")

Connection configured: True
surrealdb SDK: 3.x
SurrealDB server line: 3.x


## 3. Values read through the SDK 3.0

A model with a `datetime`, a `Duration`, a `set[str]` and an optional `str`. The record is written in SurrealQL with a
datetime to the nanosecond, a zero duration, a set and `NONE`. Read back, the datetime is a `PreciseDatetime` (a
`datetime` subclass that keeps the nanoseconds; 0.40.x truncated them to microseconds), the duration `Duration(0)`,
the set a Python `set` and the missing note `None`. Updated, the nanoseconds are written back.

In [2]:
from datetime import datetime

from surreal_orm_lite import BaseSurrealModel, Duration


class Reading(BaseSurrealModel):
    id: str | None = None
    created: datetime
    ttl: Duration
    tags: set[str]
    note: str | None = None


await client.query(
    "CREATE Reading:r1 SET created = d'2026-11-06T12:00:00.123456789Z', ttl = 0ns, "
    "tags = <set> ['b', 'a', 'b'], note = NONE;"
)
reading = await Reading.objects().get("r1")
print(type(reading.created).__name__, reading.created.microsecond, reading.created.nanosecond)
print(repr(reading.ttl), sorted(reading.tags), reading.note)

await reading.update()  # written back with its nanoseconds
print((await Reading.objects().get("r1")).created.nanosecond)

PreciseDatetime 123456 789
Duration(elapsed=0) ['a', 'b'] None
789


## 4. `raw_query()` returns the rows of every statement

`raw_query()` and `QuerySet.query()` run every statement and return the rows of every one, in order, as the full
SurrealDB-ORM does (0.40.x returned the first statement's rows only, with a warning). A `LET` adds no row.

In [3]:
class Reader(BaseSurrealModel):
    id: str | None = None
    name: str
    age: int


await Reader(id="ada", name="Ada", age=36).save()
await Reader(id="bob", name="Bob", age=17).save()
rows = await Reader.raw_query(
    "LET $min = 18; SELECT * FROM Reader WHERE age >= $min; SELECT * FROM Reader WHERE age < $min;"
)
print([reader.name for reader in rows])

['Ada', 'Bob']


## 5. `first()` and pages

On SurrealDB 3.x, 4.x and the embedded engine, `first()` without `order_by()` ends its statement with `ORDER BY id`:
the records below are created in shuffled order and `first()` returns the smallest id (on 2.x the statement is
unchanged and a plain scan already reads in id order). Pages (`all()`, `limit()`/`offset()`) without `order_by()` stay
unordered — SurrealDB 4.x returns rows in no fixed order without `ORDER BY` — so add `order_by()` to page.

In [4]:
for n in (3, 1, 2):
    await Reader(id=f"r{n}", name=f"R{n}", age=20 + n).save()
print("first():", (await Reader.objects().filter(name__startswith="R").first()).get_raw_id())

page = await Reader.objects().filter(name__startswith="R").order_by("id").limit(2).offset(1).exec()
print("page 2 of 2-row pages:", [reader.get_raw_id() for reader in page])

first(): r1
page 2 of 2-row pages: ['r2', 'r3']


## 6. Transactions

`transaction()` picks its strategy from the server: over WebSocket on SurrealDB 3.x and 4.x an
`InteractiveTransaction` (reads inside see the uncommitted writes); on SurrealDB 2.x, over HTTP and on the embedded
engine a `BufferedTransaction` (writes flushed as one `BEGIN … COMMIT` at the end of the block).

In [5]:
async with SurrealDBConnectionManager.transaction() as tx:
    await Reader(id="tx1", name="InTx", age=40).save(tx=tx)
    print(f"SurrealDB {LINE}.x:", type(tx).__name__)
print("committed:", await Reader.objects().filter(name="InTx").count())

SurrealDB 3.x: InteractiveTransaction
committed: 1


## 7. The embedded engine is SurrealDB 4.0

With the `embedded` extra the same models run in process: here a second connection, `embedded`, on `mem://`. The
engine is a SurrealDB 4.0 core (it was 2.3.10 with SDK 2.0), so the ORM takes its 4.x branches there.

Its session starts as the in-process owner. After a record user signs up, `invalidate()` returns the session to the
owner, and the owner sees the data intact.

In [6]:
SurrealDBConnectionManager.add_connection("embedded", url="mem://", namespace="examples", database="sdk3_demo")

async with SurrealDBConnectionManager.using("embedded"):
    print("embedded:", SurrealDBConnectionManager.is_embedded())
    embedded = await SurrealDBConnectionManager.get_client()
    print(f"engine line: {await cached_major(embedded)}.x")
    await Reader(id="ada", name="Ada", age=36).save()

    await embedded.query(
        "DEFINE TABLE app_user SCHEMALESS PERMISSIONS FOR select WHERE id = $auth.id; "
        "DEFINE ACCESS account ON DATABASE TYPE RECORD "
        "SIGNUP (CREATE app_user SET email = $email, pass = crypto::argon2::generate($pass)) "
        "SIGNIN (SELECT * FROM app_user WHERE email = $email AND crypto::argon2::compare(pass, $pass));"
    )
    await SurrealDBConnectionManager.signup(access="account", variables={"email": "ada@example.com", "pass": "s3cret"})
    print("signed in as a record user:", (await SurrealDBConnectionManager.info())["email"])
    await SurrealDBConnectionManager.invalidate()
    print("back to the owner, readers:", await Reader.objects().count())

embedded: True
engine line: 4.x


signed in as a record user: ada@example.com


back to the owner, readers: 1


## 8. Migrations on SurrealDB 4.x

SurrealDB 4.x creates its databases at compatibility level 4, where a field of a populated `SCHEMAFULL` table can only
be redefined with a type that accepts every old value, and a new field must accept `NONE`. A migration that narrows
`person.age` from `int` to `string` on a table that holds rows raises `FieldChangeRefusedError` (a
`MigrationStatementError` naming the `table` and the `field`): it is refused inside the migration's transaction,
before the field change runs, and nothing is applied. The same change written as a `RawSQL` with SurrealDB 4.x's
`MIGRATE EXISTING ROWS … FALLBACK …` converts the rows. SurrealDB 2.x and 3.x apply the `AlterField` as before.

`field_change()` runs the scenario on one connection; it runs below on the connected server, then on the embedded
engine (always 4.x).

In [7]:
import tempfile
from pathlib import Path

from surreal_orm_lite.migrations import FieldChangeRefusedError, MigrationExecutor

INITIAL = '''
from surreal_orm_lite.migrations import AddField, CreateTable, Migration

migration = Migration(
    name="0001_person",
    operations=[CreateTable(name="person"), AddField(table="person", name="age", field_type="int")],
)
'''

AGE_AS_STRING = '''
from surreal_orm_lite.migrations import AlterField, Migration

migration = Migration(
    name="0002_age_as_string",
    dependencies=["0001_person"],
    operations=[AlterField(table="person", name="age", field_type="string")],
)
'''

AGE_AS_STRING_4X = '''
from surreal_orm_lite.migrations import AlterField, Migration, RawSQL

migration = Migration(
    name="0002_age_as_string",
    dependencies=["0001_person"],
    operations=[
        RawSQL(
            "DEFINE FIELD OVERWRITE age ON person TYPE string "
            "MIGRATE EXISTING ROWS AS age_text USING <string> $value FALLBACK '';",
            state_operations=[AlterField(table="person", name="age", field_type="string")],
        )
    ],
)
'''


async def ages(connection: str) -> list:
    db = await SurrealDBConnectionManager.get_client(connection)
    [rows] = await db.query("SELECT id, age FROM person ORDER BY id;")
    return [row["age"] for row in rows]


async def field_change(connection: str) -> None:
    """Narrow person.age from int to string on a populated SCHEMAFULL table of *connection*."""
    directory = Path(tempfile.mkdtemp())
    executor = MigrationExecutor(directory, connection=connection)
    (directory / "0001_person.py").write_text(INITIAL)
    print("migrate():", await executor.migrate())
    db = await SurrealDBConnectionManager.get_client(connection)
    await db.query("CREATE person:ada SET age = 36; CREATE person:bob SET age = 17;")

    (directory / "0002_age_as_string.py").write_text(AGE_AS_STRING)
    try:
        applied = await executor.migrate()
    except FieldChangeRefusedError as error:
        print(f"FieldChangeRefusedError: table={error.table!r} field={error.field!r}")
        print("  ", error)
    else:
        # SurrealDB 2.x and 3.x: the field is redefined, the rows keep the values they hold.
        print("AlterField applied:", applied)
        print("rows as written, ages:", await ages(connection))
        return
    print("nothing applied, ages:", await ages(connection))

    (directory / "0002_age_as_string.py").write_text(AGE_AS_STRING_4X)
    print("RawSQL applied:", await executor.migrate())
    print("rows converted, ages:", await ages(connection))

In [8]:
print(f"-- the connected server, SurrealDB {LINE}.x")
await field_change("default")

-- the connected server, SurrealDB 3.x
migrate(): ['0001_person']
AlterField applied: ['0002_age_as_string']
rows as written, ages: [36, 17]


In [9]:
print("-- the embedded engine, SurrealDB 4.0")
await field_change("embedded")

-- the embedded engine, SurrealDB 4.0
migrate(): ['0001_person']
FieldChangeRefusedError: table='person' field='age'
   Migration '0002_age_as_string', operation 1 (Alter field age on person): SurrealDB 4.x refuses this change because table 'person' is SCHEMAFULL, holds rows, and its database is at compatibility level 4. On such a table a field can only be redefined with a type that accepts every value of the old one, and a new field must accept NONE. Nothing was applied. Write the change as a RawSQL operation that converts the existing rows (SurrealDB 4.x: `DEFINE FIELD … MIGRATE EXISTING ROWS AS <name> USING <expression> FALLBACK <value>`), with `state_operations=[…]` describing the result; a MigrateRows operation is planned for v0.56.0.
nothing applied, ages: [36, 17]
RawSQL applied: ['0002_age_as_string']
rows converted, ages: ['36', '17']


## 9. Supported lines and release lines

The ORM supports the server lines its SDK supports; the SDK refuses a server outside 2.0.0 to 5.0.0 (excluded) at
connect time.

In [10]:
print("Supported SurrealDB lines: 2.x, 3.x, 4.x (tested on 2.7.0, 3.3.5 and 4.0.0)")
print()
print(f"{'surreal-orm-lite':<18}{'official SDK':<16}status")
print(f"{'0.50.x':<18}{'3.0':<16}current")
print(f"{'0.40.x':<18}{'2.0':<16}deprecated, security fixes only: pip install \"surreal-orm-lite<0.50\"")

Supported SurrealDB lines: 2.x, 3.x, 4.x (tested on 2.7.0, 3.3.5 and 4.0.0)

surreal-orm-lite  official SDK    status
0.50.x            3.0             current
0.40.x            2.0             deprecated, security fixes only: pip install "surreal-orm-lite<0.50"


## 10. Cleanup

In [11]:
await client.query("REMOVE DATABASE IF EXISTS sdk3_demo;")
await SurrealDBConnectionManager.remove_connection("embedded")
await SurrealDBConnectionManager.close_connection()
print("Cleaned up")

Cleaned up
